In [0]:
from datetime import datetime 
from pyspark.sql.functions import lit, col, to_date

#User pass date 
try:
	arrival_date=dbutils.widgets.get("arrival_date")
except Exception:
	arrival_date=datetime.now().strftime("%Y-%m-%d")

#User pass catalog
try:
	catalog=dbutils.widgets.get("catalog")
except Exception:
	catalog="travel_bookings"

#User pass schema
try:
	schema=dbutils.widgets.get("schema")
except Exception:
	schema="default"

In [0]:
#Read bronze layer customer table
df_customer=spark.read.table(f"{catalog}.bronze.customer").filter(col("business_date")==datetime.strptime(arrival_date, "%Y-%m-%d").date())

from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite, VerificationResult 

#Create checks using pydeequ library
check=Check(spark, CheckLevel.Error, "Data Quality checks")\
.hasSize(lambda x: x > 0)\
.isComplete("customer_name")\
.isComplete("customer_address")\
.isComplete("email")

#Run check against dataframe
result=VerificationSuite(spark)\
.onData(df_customer)\
.addCheck(check)\
.run()

df=VerificationResult.checkResultsAsDataFrame(spark, result)

#Create schema 
spark.sql(f"create schema if not exists {catalog}.ops")

#Create table 
spark.sql(f"""create table if not exists {catalog}.ops.dq_results 
(
business_date date, dataset string, check_name string, status string,
constraint string, message string, recorded_at timestamp
) using delta
""")

out=df.withColumn("business_date", to_date(lit(arrival_date))).withColumn("dataset", lit("customer_inc"))

#Write into results table
out.select("business_date", "dataset", "check", "check_status", "constraint",
"constraint_status", "constraint_message", "recorded_at").write.mode("append")\
.option("mergeSchema", "true").saveAsTable(f"{catalog}.ops.dq_results")

if result.status!="SUCCESS":
	raise ValueError("DQ failed for customers")
else:
	print("Customer DQ passed")